In [1]:
import sqlite3

conn = sqlite3.connect("lahman_1871-2022.sqlite")
cursor = conn.cursor()

In [12]:
cursor.execute("""
SELECT name
FROM sqlite_master
WHERE type='table';
""")

print(cursor.fetchall())

[('AllstarFull',), ('Appearances',), ('AwardsManagers',), ('AwardsPlayers',), ('AwardsShareManagers',), ('AwardsSharePlayers',), ('Batting',), ('BattingPost',), ('CollegePlaying',), ('Fielding',), ('FieldingOF',), ('FieldingOFsplit',), ('FieldingPost',), ('HallOfFame',), ('HomeGames',), ('Managers',), ('ManagersHalf',), ('Parks',), ('People',), ('Pitching',), ('PitchingPost',), ('Salaries',), ('Schools',), ('SeriesPost',), ('Teams',), ('TeamsFranchises',), ('TeamsHalf',)]


# Predict Player Salary from Batting Statistics

In [17]:
import pandas as pd

query = """
SELECT
    yearID,
    teamID,
    R,
    H,
    HR,
    BB,
    SO,
    ERA,
    W
FROM Teams
WHERE yearID > 1990
"""

df = pd.read_sql(query, conn)
df

,yearID,teamID,R,H,HR,BB,SO,ERA,W
0,1991,BAL,686,1421,170,528,974,4.59,67
1,1991,BOS,731,1486,126,593,820,4.01,84
2,1991,CAL,653,1396,115,448,928,3.69,81
3,1991,CHA,758,1464,139,610,896,3.79,87
4,1991,CLE,576,1390,79,449,888,4.23,57
...,...,...,...,...,...,...,...,...,...
937,2022,PIT,591,1186,158,476,1497,4.66,62
938,2022,SDN,705,1317,153,574,1327,3.81,89
939,2022,SFN,716,1261,183,571,1462,3.85,81
940,2022,SLN,772,1386,197,537,1226,3.79,93


In [18]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

X = df[["R","H","HR","BB","SO","ERA"]]
y = df["W"]

X_train, X_test, y_train, y_test = train_test_split(
    X,y,test_size=0.2,random_state=42
)

model = RandomForestRegressor(n_estimators=200)
model.fit(X_train,y_train)

preds = model.predict(X_test)

print("R2:", r2_score(y_test,preds))

R2: 0.8947174027972167


In [19]:
importance = pd.DataFrame({
    "feature": X.columns,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

print(importance)

  feature  importance
0       R    0.347500
5     ERA    0.288227
1       H    0.260322
3      BB    0.074343
4      SO    0.016521
2      HR    0.013086


# Predict Hall of Fame Induction

In [20]:
import pandas as pd

query = """
SELECT
    b.playerID,
    SUM(b.H) AS career_hits,
    SUM(b.HR) AS career_hr,
    SUM(b.RBI) AS career_rbi,
    SUM(b.BB) AS career_walks,
    CASE
        WHEN MAX(CASE WHEN h.inducted = 'Y' THEN 1 ELSE 0 END) = 1
        THEN 1
        ELSE 0
    END AS hof
FROM Batting b
LEFT JOIN HallOfFame h
    ON b.playerID = h.playerID
GROUP BY b.playerID
HAVING career_hits > 500
"""

df = pd.read_sql(query, conn)
df

,playerID,career_hits,career_hr,career_rbi,career_walks,hof
0,aaronha01,3771,755,2297,1402,1
1,abbated01,772,11,324,289,0
2,abbotku01,523,62,242,133,0
3,abreubo01,7410,864,4089,4428,0
4,abreujo02,1445,243,863,386,0
...,...,...,...,...,...,...
2681,zimmehe01,1566,58,796,242,0
2682,zimmery01,1846,284,1061,646,0
2683,ziskri01,1477,207,792,533,0
2684,zobribe01,1566,167,768,832,0


In [21]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

X = df[[
    "career_hits",
    "career_hr",
    "career_rbi",
    "career_walks"
]]

y = df["hof"]

X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size=0.2,stratify=y,random_state=42
)

model = RandomForestClassifier(n_estimators=200)
model.fit(X_train,y_train)

preds = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test,preds))

Accuracy: 0.9219330855018587


In [22]:
importance = pd.DataFrame({
    "feature": X.columns,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

print(importance)

        feature  importance
0   career_hits    0.315608
2    career_rbi    0.309274
3  career_walks    0.213949
1     career_hr    0.161168
